# Cynbit Lead-to-Proposal AI Copilot — Interactive Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yadavabhi2828-beep/cynbit-lead-copilot/blob/main/demo.ipynb)

This notebook demonstrates the end-to-end pipeline of the **Lead-to-Proposal AI Copilot** developed for the Cynbit AI/ML Engineer Intern assessment.

### Pipeline Steps:
1. **Extract**: Parses intent, service line, budget, timeline, and urgency (supports English and Hinglish).
2. **Score**: Transparent, explainable 5-factor scoring rubric (0–100) assigning HOT/WARM/COLD tiers and response SLAs.
3. **Retrieve**: Grounds recommendations in Cynbit's actual services, products, and 13 case studies via TF-IDF retrieval.
4. **Draft**: Drafts a contextual proposal response and discovery questions with guardrails.

## 1. Setup & Environment
Clone the repository and install dependencies.

In [ ]:
import os
if not os.path.exists("copilot"):
    !git clone https://github.com/yadavabhi2828-beep/cynbit-lead-copilot.git
    %cd cynbit-lead-copilot
!pip install -q -r requirements.txt


## 2. Run Test Suite
Verify the full suite of 40 unit and regression tests.

In [1]:
!pytest -q

........................................                                 [100%]
=============================== warnings summary ===============================
tests/test_pipeline.py::test_api_triage
  /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/starlette/formparsers.py:12: PendingDeprecationWarning: Please use `import python_multipart` instead.
    import multipart

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
40 passed, 1 warning in 0.97s


## 3. Run Single Lead Triage
Process a sample healthcare AI enquiry end-to-end without needing any API keys (offline baseline mode).

In [2]:
from copilot.pipeline import LeadCopilot
from copilot.schemas import Lead
import pandas as pd

# Initialize copilot (offline mode uses rules + TF-IDF retrieval + templates)
copilot = LeadCopilot(use_llm=False)

# Sample enquiry
lead = Lead(
    name="Dr. Priya Sharma",
    company="MediCore Health",
    email="priya@medicore.in",
    message="We need an AI assistant for patient appointment reminders and WhatsApp follow-ups. Budget is approx ₹8 lakhs, timeline within 6 weeks. Please send a proposal."
)

res = copilot.process(lead)

print(f"\n{'='*50}")
print(f"  TRIAGE SUMMARY")
print(f"{'='*50}")
print(f"Tier:            {res.score.tier.value}")
print(f"Score:           {res.score.score}/100")
print(f"Intent:          {res.extraction.intent.value}")
print(f"Service Line:    {res.extraction.service_line.value}")
print(f"Route To:        {res.score.route.value}")
print(f"Recommended SLA: {res.score.recommended_sla}")

print(f"\n{'='*50}")
print(f"  SCORE BREAKDOWN (EXPLAINABLE RUBRIC)")
print(f"{'='*50}")
print(pd.DataFrame([c.model_dump() for c in res.score.components]).to_string(index=False))

print(f"\n{'='*50}")
print(f"  GROUNDING EVIDENCE (FROM CYNBIT KNOWLEDGE BASE)")
print(f"{'='*50}")
for e in res.evidence:
    print(f"• [{e.type.upper()}] {e.title} (similarity: {e.score:.2f}) -> {e.url}")

print(f"\n{'='*50}")
print(f"  DRAFT PROPOSAL REPLY (FOR HUMAN REVIEW)")
print(f"{'='*50}")
if res.proposal:
    print(f"Subject: {res.proposal.subject}\n")
    print(res.proposal.body)
    print(f"\nDiscovery Questions:")
    for q in res.proposal.discovery_questions:
        print(f"  - {q}")


  TRIAGE SUMMARY
Tier:            HOT
Score:           77/100
Intent:          new_project
Service Line:    ai_automation
Route To:        sales
Recommended SLA: Respond within 1 hour; assign a senior owner.

  SCORE BREAKDOWN (EXPLAINABLE RUBRIC)
           name  points  max_points                                                     reason
         budget      25          25                              Strong budget (~INR 800,000).
        urgency      12          20                         Urgency medium ('within 6 weeks').
    specificity       5          20                1 relevant requirement signal(s); 25 words.
intent_strength      20          20 Intent 'new_project'; explicit ask for proposal/demo/call.
contact_quality      15          15                     Company given; business e-mail domain.

  GROUNDING EVIDENCE (FROM CYNBIT KNOWLEDGE BASE)
• [SERVICE] Autonomous AI agents & cognitive automation (similarity: 0.17) -> https://www.cynbit.in/services
• [SERVICE] CRM autom

## 4. Batch Triage on Sample Leads
Process dev and holdout sample leads and rank the sales queue by lead score.

In [3]:
import json
from copilot.config import DATA_DIR

samples = json.loads((DATA_DIR / "sample_leads.json").read_text(encoding="utf-8")) + \
          json.loads((DATA_DIR / "holdout_leads.json").read_text(encoding="utf-8"))

queue = []
for s in samples:
    r = copilot.process(Lead(**s["lead"]))
    queue.append({
        "ID": s["id"],
        "Company / Name": s["lead"]["company"] or s["lead"]["name"],
        "Tier": r.score.tier.value,
        "Score": r.score.score,
        "Intent": r.extraction.intent.value,
        "Service": r.extraction.service_line.value,
        "Route": r.score.route.value,
        "SLA": r.score.recommended_sla
    })

df_queue = pd.DataFrame(queue).sort_values("Score", ascending=False)
display(df_queue.head(10))

     ID  ...                                            SLA
8   L09  ...  Respond within 1 hour; assign a senior owner.
0   L01  ...  Respond within 1 hour; assign a senior owner.
11  L12  ...  Respond within 1 hour; assign a senior owner.
3   L04  ...  Respond within 1 hour; assign a senior owner.
12  L13  ...  Respond within 1 hour; assign a senior owner.
1   L02  ...                 Respond the same business day.
20  H04  ...                 Respond the same business day.
15  L16  ...                 Respond the same business day.
25  H09  ...                 Respond the same business day.
9   L10  ...                 Respond the same business day.

[10 rows x 8 columns]


## 5. Run Evaluation Script
Run the benchmark evaluation across dev and held-out sets.

In [4]:
!python eval/run_eval.py
!cat eval/RESULTS.md

# Evaluation results (offline deterministic baseline)

## dev set (sample_leads.json)

| metric | value |
|---|---|
| n_leads | 17 |
| intent_accuracy | 1.0 |
| tier_accuracy | 1.0 |
| service_line_accuracy | 1.0 |
| retrieval_hit_rate | 1.0 |

| id | expected intent | predicted intent | expected tier | predicted tier | score | status |
|---|---|---|---|---|---|---|
| L01 | new_project | new_project | HOT | HOT | 87 | OK |
| L02 | new_project | new_project | WARM | WARM | 62 | OK |
| L03 | new_project | new_project | COLD | COLD | 28 | OK |
| L04 | product_demo | product_demo | HOT | HOT | 75 | OK |
| L05 | training | training | WARM | WARM | 48 | OK |
| L06 | career | career | NOT_A_PROSPECT | NOT_A_PROSPECT | 0 | OK |
| L07 | vendor_solicitation | vendor_solicitation | NOT_A_PROSPECT | NOT_A_PROSPECT | 0 | OK |
| L08 | spam_or_irrelevant | spam_or_irrelevant | NOT_A_PROSPECT | NOT_A_PROSPECT | 0 | OK |
| L09 | new_project | new_project | HOT | HOT | 95 | OK |
| L10 | new_project | ne